# PubMed Literature Harvester

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook searches PubMed for academic literature and enriches the results with metadata from multiple sources. It transforms a search query into a research-ready dataset containing titles, authors, abstracts, citations, DOIs, and full-text links where available.

The notebook consolidates data from six different APIs (PubMed, CrossRef, OpenAlex, Semantic Scholar, Europe PMC, and NCBI ID Converter) into a single enrichment pipeline, attempting each source in priority order to maximize data completeness.

## Key Features

- **Multi-Source Enrichment**: Automatically fills missing DOIs, abstracts, and citations from multiple APIs
- **PMC Full-Text Linking**: Identifies open-access articles and generates direct links to full text
- **Citation Metrics**: Retrieves citation counts from OpenAlex for impact assessment
- **Batch Processing**: Efficiently handles large result sets with rate limiting and progress tracking
- **Data Exploration**: Built-in summaries of publication years, top journals, and citation statistics
- **Filtered Exports**: Export subsets by full-text availability or citation impact

## Workflow

1. **Configure**: Set your PubMed search query and parameters
2. **Search**: Retrieve matching articles from PubMed
3. **Enrich**: Fill missing metadata from CrossRef, OpenAlex, Semantic Scholar, and Europe PMC
4. **Explore**: Review publication trends and citation statistics
5. **Export**: Download clean CSV and JSONL files for further analysis

## Applications

This notebook supports systematic literature reviews, bibliometric analysis, and research mapping in health-related fields. It is particularly useful for medical anthropology, public health research, and any study that draws on biomedical literature. The structured exports integrate with reference managers and qualitative analysis tools.

## Methodological Positioning

This notebook represents a **computational anthropology** approach to literature engagement. The enrichment pipeline maximizes metadata completeness, but the researcher's disciplinary judgment remains essential for evaluating relevance and quality of sources.

**Important**: PubMed indexes primarily biomedical and life sciences literature. For broader social science coverage, combine with the Google Scholar Explorer notebook.

## Target Audience

Designed for anthropologists and qualitative researchers working with biomedical and health sciences literature, from medical anthropology dissertation research to systematic reviews and applied health projects.

## Technical Approach

The notebook uses NCBI E-utilities (PubMed's official API) for search and retrieval, supplemented by CrossRef, OpenAlex, Semantic Scholar, and Europe PMC APIs for metadata enrichment. No API key is required for basic use, though an optional NCBI API key increases rate limits. All processing runs in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2025. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

*Install required Python packages and import necessary libraries for API access, data processing, and progress tracking. Run this cell first to ensure all dependencies are available.*

In [ ]:
!pip -q install pandas tqdm requests beautifulsoup4 lxml

import os
import time
import math
import json
import ast
import urllib.parse
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field, asdict

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
import xml.etree.ElementTree as ET

print("✓ Environment ready.")

## Configuration

*Set your search parameters, API credentials, and output preferences. The email is required by NCBI for polite API usage; the API key is optional but increases rate limits.*

In [ ]:
# =============================================================================
# USER CONFIGURATION
# =============================================================================

# Required: Your email for NCBI API (be polite!)
EMAIL = "you@example.com"  #@param {type:"string"}

# Optional: NCBI API key (get one at https://ncbiinsights.ncbi.nlm.nih.gov/2017/11/02/new-api-keys-for-the-e-utilities/)
API_KEY = ""  #@param {type:"string"}

# Search query - uses PubMed syntax
# Examples:
#   '"integrative medicine"[Title]' - phrase in title only
#   'anthropology AND artificial intelligence' - both terms anywhere
#   '"climate change"[Title/Abstract]' - phrase in title or abstract
QUERY = "\"COVID\" \"Testosterone\""  #@param {type:"string"}

# Maximum records to retrieve (None = all matching records)
MAX_RECORDS = 345  #@param {type:"integer"}

# Date filtering (optional)
DATETYPE = "pubdate"  #@param ["pubdate", "edat"]
MINDATE = None  # Format: "2020/01/01" or None
MAXDATE = None  # Format: "2024/12/31" or None

# Output settings
OUTPUT_DIR = "."  #@param {type:"string"}
OUTPUT_PREFIX = "pubmed_results"  #@param {type:"string"}

# =============================================================================
# INTERNAL SETTINGS (modify if needed)
# =============================================================================
NCBI_TOOL = "pubmed_harvester"
SLEEP_SECS = 0.34 if not API_KEY else 0.11  # NCBI rate limits
EFETCH_BATCH_SIZE = 200
ELINK_BATCH_SIZE = 200

print(f"✓ Configuration loaded")
print(f"  Query: {QUERY}")
print(f"  Output: {OUTPUT_DIR}/{OUTPUT_PREFIX}_*.csv")

## Core Functions

*Define the enrichment engine that consolidates all API sources into a unified data retrieval system. These functions handle PubMed search, metadata enrichment, and data cleaning.*

In [ ]:
# =============================================================================
# DATA STRUCTURES
# =============================================================================

@dataclass
class Article:
    """Unified article record with all metadata fields."""
    pmid: str = ""
    title: str = ""
    authors: List[str] = field(default_factory=list)
    journal: str = ""
    journal_iso: str = ""
    publication_date: str = ""
    year: str = ""
    doi: str = ""
    abstract: str = ""
    citations: int = 0
    pmcid: str = ""
    full_text_url: str = ""

# =============================================================================
# NCBI E-UTILITIES
# =============================================================================

EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

def _ncbi_params() -> Dict[str, Any]:
    """Common parameters for NCBI requests."""
    p = {"tool": NCBI_TOOL}
    if EMAIL:
        p["email"] = EMAIL
    if API_KEY:
        p["api_key"] = API_KEY
    return p

def ncbi_get(endpoint: str, params: Dict[str, Any], max_retries: int = 5) -> requests.Response:
    """Make NCBI API request with rate limiting and retry logic."""
    url = EUTILS_BASE + endpoint
    merged = {**params, **_ncbi_params()}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=merged, timeout=60)
            if r.status_code == 200:
                time.sleep(SLEEP_SECS)
                return r
            time.sleep(SLEEP_SECS + min(2**attempt, 10))
        except requests.RequestException:
            time.sleep(SLEEP_SECS + min(2**attempt, 10))

    r.raise_for_status()
    return r

def chunked(lst: List[Any], n: int):
    """Yield successive n-sized chunks from list."""
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# =============================================================================
# PUBMED SEARCH
# =============================================================================

def search_pubmed(term: str, max_records: Optional[int] = None,
                  datetype: Optional[str] = None, mindate: Optional[str] = None,
                  maxdate: Optional[str] = None) -> Tuple[List[str], int]:
    """Search PubMed and return list of PMIDs."""
    pmids = []
    retmax = 10000

    qparams = {
        "db": "pubmed",
        "term": term,
        "retmode": "json",
        "retstart": 0,
        "retmax": retmax,
    }
    if datetype:
        qparams["datetype"] = datetype
    if mindate:
        qparams["mindate"] = mindate
    if maxdate:
        qparams["maxdate"] = maxdate

    # Get initial count
    r = ncbi_get("esearch.fcgi", qparams)
    j = r.json()
    total_count = int(j["esearchresult"]["count"])
    target = min(total_count, max_records) if max_records else total_count

    pbar = tqdm(total=target, desc="Searching PubMed")
    while len(pmids) < target:
        qparams["retstart"] = len(pmids)
        qparams["retmax"] = min(retmax, target - len(pmids))
        r = ncbi_get("esearch.fcgi", qparams)
        ids = r.json()["esearchresult"].get("idlist", [])
        pmids.extend(ids)
        pbar.update(len(ids))
        if not ids:
            break
    pbar.close()

    return pmids[:target], total_count

# =============================================================================
# PUBMED XML PARSING
# =============================================================================

def _get_text(elem: Optional[ET.Element]) -> str:
    return (elem.text or "").strip() if elem is not None else ""

def _parse_date(article: ET.Element) -> Tuple[str, str]:
    """Extract and normalize publication date, return (raw_date, year)."""
    months = {'Jan':'01','Feb':'02','Mar':'03','Apr':'04','May':'05','Jun':'06',
              'Jul':'07','Aug':'08','Sep':'09','Oct':'10','Nov':'11','Dec':'12'}

    year = article.findtext(".//PubDate/Year") or ""
    month = article.findtext(".//PubDate/Month") or ""
    day = article.findtext(".//PubDate/Day") or ""

    if not year:
        medline = article.findtext(".//PubDate/MedlineDate") or ""
        if medline:
            year = medline[:4] if len(medline) >= 4 else ""
            return medline, year

    # Normalize month
    month_num = months.get(month, month.zfill(2) if month.isdigit() else "01")
    day_num = day.zfill(2) if day else "01"

    if year:
        date_str = f"{year}-{month_num}-{day_num}"
        return date_str, year

    return "", ""

def _parse_authors(article: ET.Element) -> List[str]:
    """Extract author names from PubMed XML."""
    authors = []
    for a in article.findall(".//Author"):
        collective = a.findtext("CollectiveName")
        if collective:
            authors.append(collective.strip())
            continue
        last = a.findtext("LastName") or ""
        fore = a.findtext("ForeName") or a.findtext("Initials") or ""
        if last:
            authors.append(f"{last}, {fore}" if fore else last)
    return authors

def _parse_abstract(article: ET.Element) -> str:
    """Extract abstract text, handling structured abstracts."""
    absnode = article.find(".//Abstract")
    if absnode is None:
        return ""
    parts = []
    for elem in absnode.findall("AbstractText"):
        label = elem.get("Label")
        txt = (elem.text or "").strip()
        if label and txt:
            parts.append(f"{label}: {txt}")
        elif txt:
            parts.append(txt)
    return "\n\n".join(parts)

def _parse_doi(article: ET.Element) -> str:
    """Extract DOI from article XML."""
    for idn in article.findall(".//ArticleId"):
        if idn.get("IdType", "").lower() == "doi":
            return (idn.text or "").strip()
    for eloc in article.findall(".//ELocationID"):
        if eloc.get("EIdType", "").lower() == "doi":
            return (eloc.text or "").strip()
    return ""

def fetch_pubmed_details(pmids: List[str]) -> List[Article]:
    """Fetch article details from PubMed for given PMIDs."""
    articles = []

    for batch in tqdm(list(chunked(pmids, EFETCH_BATCH_SIZE)), desc="Fetching PubMed records"):
        params = {"db": "pubmed", "id": ",".join(batch), "retmode": "xml"}
        r = ncbi_get("efetch.fcgi", params)
        root = ET.fromstring(r.content)

        for art in root.findall(".//PubmedArticle"):
            date_str, year = _parse_date(art)
            articles.append(Article(
                pmid=_get_text(art.find(".//PMID")),
                title=_get_text(art.find(".//ArticleTitle")),
                authors=_parse_authors(art),
                journal=_get_text(art.find(".//Journal/Title")),
                journal_iso=_get_text(art.find(".//ISOAbbreviation")),
                publication_date=date_str,
                year=year,
                doi=_parse_doi(art),
                abstract=_parse_abstract(art),
            ))

    return articles

# =============================================================================
# ENRICHMENT: DOI LOOKUP
# =============================================================================

def lookup_doi_crossref(title: str) -> Optional[str]:
    """Search CrossRef for DOI by title."""
    if not title:
        return None

    clean = title.replace('[', '').replace(']', '').strip()
    query = urllib.parse.quote(clean)
    url = f"https://api.crossref.org/works?query.title={query}&rows=1"
    headers = {'User-Agent': f'PubMedHarvester/1.0 (mailto:{EMAIL})'}

    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            items = r.json().get('message', {}).get('items', [])
            if items:
                return items[0].get('DOI')
    except Exception:
        pass
    return None

# =============================================================================
# ENRICHMENT: ABSTRACT LOOKUP (cascading sources)
# =============================================================================

def lookup_abstract_openalex(doi: str) -> Optional[str]:
    """Fetch abstract from OpenAlex using DOI."""
    if not doi:
        return None

    url = f"https://api.openalex.org/works/https://doi.org/{doi}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            index = r.json().get('abstract_inverted_index')
            if index:
                word_pos = [(pos, word) for word, positions in index.items() for pos in positions]
                word_pos.sort()
                return " ".join(w for _, w in word_pos)
    except Exception:
        pass
    return None

def lookup_abstract_semantic_scholar(doi: str, pmid: str) -> Optional[str]:
    """Fetch abstract from Semantic Scholar using DOI or PMID."""
    for id_type, id_val in [("DOI", doi), ("PMID", pmid)]:
        if not id_val:
            continue
        try:
            url = f"https://api.semanticscholar.org/graph/v1/paper/{id_type}:{id_val}?fields=abstract"
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                abstract = r.json().get('abstract')
                if abstract:
                    return abstract
        except Exception:
            pass
    return None

def lookup_abstract_europepmc(doi: str, pmid: str) -> Optional[str]:
    """Fetch abstract from Europe PMC."""
    query = f'DOI:"{doi}"' if doi else f'EXT_ID:"{pmid}"' if pmid else None
    if not query:
        return None

    try:
        url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
        params = {'query': query, 'resultType': 'core', 'format': 'json'}
        r = requests.get(url, params=params, timeout=5)
        if r.status_code == 200:
            results = r.json().get('resultList', {}).get('result', [])
            if results:
                return results[0].get('abstractText')
    except Exception:
        pass
    return None

def lookup_abstract_cascade(doi: str, pmid: str) -> Optional[str]:
    """Try multiple sources to find abstract."""
    for lookup_fn in [
        lambda: lookup_abstract_openalex(doi),
        lambda: lookup_abstract_semantic_scholar(doi, pmid),
        lambda: lookup_abstract_europepmc(doi, pmid)
    ]:
        result = lookup_fn()
        if result:
            return result
        time.sleep(0.1)
    return None

# =============================================================================
# ENRICHMENT: CITATIONS & PMCID
# =============================================================================

def lookup_citations_openalex(doi: str) -> Optional[int]:
    """Fetch citation count from OpenAlex."""
    if not doi:
        return None

    try:
        url = f"https://api.openalex.org/works/https://doi.org/{doi}"
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            return r.json().get('cited_by_count', 0)
    except Exception:
        pass
    return None

def lookup_pmcids_batch(pmids: List[str]) -> Dict[str, str]:
    """Convert PMIDs to PMCIDs using NCBI ID Converter."""
    if not pmids:
        return {}

    results = {}
    url = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"

    for batch in chunked(pmids, 200):
        params = {
            'tool': NCBI_TOOL,
            'email': EMAIL,
            'ids': ",".join(batch),
            'format': 'json'
        }
        try:
            r = requests.get(url, params=params, timeout=10)
            if r.status_code == 200:
                for record in r.json().get('records', []):
                    if 'pmcid' in record:
                        results[str(record['pmid'])] = record['pmcid']
        except Exception:
            pass
        time.sleep(0.5)

    return results

# =============================================================================
# MAIN ENRICHMENT PIPELINE
# =============================================================================

def enrich_articles(articles: List[Article]) -> List[Article]:
    """Run all enrichment steps on article list."""

    # Step 1: Batch lookup PMCIDs
    print("\n📚 Looking up PMC IDs...")
    pmids = [a.pmid for a in articles if a.pmid]
    pmcid_map = lookup_pmcids_batch(pmids)

    for article in articles:
        if article.pmid in pmcid_map:
            article.pmcid = pmcid_map[article.pmid]
            article.full_text_url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{article.pmcid}/"

    print(f"   Found {len(pmcid_map)} PMC IDs")

    # Step 2: Enrich missing DOIs
    missing_doi = [a for a in articles if not a.doi and a.title]
    if missing_doi:
        print(f"\n🔍 Looking up {len(missing_doi)} missing DOIs via CrossRef...")
        for article in tqdm(missing_doi, desc="CrossRef DOI lookup"):
            doi = lookup_doi_crossref(article.title)
            if doi:
                article.doi = doi
            time.sleep(0.1)

    # Step 3: Enrich missing abstracts
    missing_abs = [a for a in articles if not a.abstract and (a.doi or a.pmid)]
    if missing_abs:
        print(f"\n📄 Looking up {len(missing_abs)} missing abstracts...")
        for article in tqdm(missing_abs, desc="Abstract lookup"):
            abstract = lookup_abstract_cascade(article.doi, article.pmid)
            if abstract:
                article.abstract = abstract

    # Step 4: Enrich citation counts
    with_doi = [a for a in articles if a.doi]
    if with_doi:
        print(f"\n📊 Fetching citation counts for {len(with_doi)} articles...")
        for article in tqdm(with_doi, desc="Citation lookup"):
            count = lookup_citations_openalex(article.doi)
            if count is not None:
                article.citations = count
            time.sleep(0.05)

    return articles

# =============================================================================
# OUTPUT FORMATTING
# =============================================================================

def articles_to_dataframe(articles: List[Article]) -> pd.DataFrame:
    """Convert article list to clean DataFrame."""
    records = []
    for a in articles:
        records.append({
            'PMID': a.pmid,
            'Title': a.title,
            'Authors': "; ".join(a.authors) if isinstance(a.authors, list) else a.authors,
            'Year': a.year,
            'PublicationDate': a.publication_date,
            'Journal': a.journal,
            'DOI': a.doi,
            'Citations': a.citations,
            'Abstract': a.abstract,
            'PMCID': a.pmcid,
            'FullTextURL': a.full_text_url,
        })

    df = pd.DataFrame(records)
    df = df.drop_duplicates(subset=['PMID'])
    return df

print("✓ Core functions loaded.")

## Run the Harvester

*Execute the main pipeline: search PubMed, fetch article details, enrich metadata from multiple sources, and save results. This cell runs all steps in sequence.*

In [ ]:
# =============================================================================
# EXECUTE PIPELINE
# =============================================================================

print("="*60)
print("PUBMED LITERATURE HARVESTER")
print("="*60)

# Step 1: Search PubMed
print(f"\n🔎 Searching PubMed for: {QUERY}")
pmids, total_count = search_pubmed(
    term=QUERY,
    max_records=MAX_RECORDS,
    datetype=DATETYPE,
    mindate=MINDATE,
    maxdate=MAXDATE
)
print(f"   Total matches: {total_count}")
print(f"   Retrieving: {len(pmids)} articles")

# Step 2: Fetch base records from PubMed
print("\n📥 Fetching article details from PubMed...")
articles = fetch_pubmed_details(pmids)
print(f"   Retrieved {len(articles)} articles")

# Step 3: Enrich with additional sources
print("\n🔄 Enriching metadata from additional sources...")
articles = enrich_articles(articles)

# Step 4: Convert to DataFrame
df = articles_to_dataframe(articles)

# Step 5: Summary statistics
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total articles:     {len(df)}")
print(f"With DOI:           {df['DOI'].notna().sum()} ({df['DOI'].notna().sum()/len(df)*100:.1f}%)")
print(f"With Abstract:      {df['Abstract'].notna().sum()} ({df['Abstract'].notna().sum()/len(df)*100:.1f}%)")
print(f"With PMC Full Text: {df['PMCID'].notna().sum()} ({df['PMCID'].notna().sum()/len(df)*100:.1f}%)")
print(f"With Citations:     {(df['Citations'] > 0).sum()} ({(df['Citations'] > 0).sum()/len(df)*100:.1f}%)")

# Step 6: Save outputs
csv_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_PREFIX}.csv")
jsonl_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_PREFIX}.jsonl")

df.to_csv(csv_path, index=False)
print(f"\n💾 Saved CSV:   {csv_path}")

# Save JSONL
with open(jsonl_path, "w", encoding="utf-8") as f:
    for article in articles:
        f.write(json.dumps(asdict(article), ensure_ascii=False) + "\n")
print(f"💾 Saved JSONL: {jsonl_path}")

# Preview
print("\n" + "="*60)
print("PREVIEW")
print("="*60)
df[['Title', 'Year', 'Citations', 'PMCID']].head(10)

## Data Exploration

*Optional: Explore the harvested data with summary statistics and visualizations. Use this section to understand your dataset before proceeding with analysis.*

In [ ]:
# =============================================================================
# OPTIONAL: DATA EXPLORATION
# =============================================================================

print("📊 DATASET OVERVIEW\n")

# Year distribution
print("Publications by Year (top 10):")
print(df['Year'].value_counts().head(10).to_string())

print("\n" + "-"*40)

# Top journals
print("\nTop 10 Journals:")
print(df['Journal'].value_counts().head(10).to_string())

print("\n" + "-"*40)

# Citation statistics
print("\nCitation Statistics:")
print(df['Citations'].describe().to_string())

print("\n" + "-"*40)

# Most cited articles
print("\nTop 5 Most Cited Articles:")
top_cited = df.nlargest(5, 'Citations')[['Title', 'Year', 'Citations', 'Journal']]
for i, row in top_cited.iterrows():
    print(f"\n  [{row['Citations']} citations] {row['Title'][:80]}...")
    print(f"    {row['Journal']} ({row['Year']})")

## Export Filtered Subsets

*Optional: Create filtered exports based on specific criteria such as year range, citation threshold, or availability of full text.*

In [ ]:
# =============================================================================
# OPTIONAL: FILTERED EXPORTS
# =============================================================================

# Example: Export only articles with abstracts and full text available
df_full_text = df[df['PMCID'].notna() & df['Abstract'].notna()]
print(f"Articles with full text AND abstract: {len(df_full_text)}")

if len(df_full_text) > 0:
    export_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_PREFIX}_fulltext.csv")
    df_full_text.to_csv(export_path, index=False)
    print(f"💾 Saved: {export_path}")

# Example: Export highly cited articles (top 10%)
citation_threshold = df['Citations'].quantile(0.9)
df_highly_cited = df[df['Citations'] >= citation_threshold]
print(f"\nHighly cited articles (≥{citation_threshold:.0f} citations): {len(df_highly_cited)}")

if len(df_highly_cited) > 0:
    export_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_PREFIX}_highly_cited.csv")
    df_highly_cited.to_csv(export_path, index=False)
    print(f"💾 Saved: {export_path}")